# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.706134531139746)),
 (1, np.float64(1.706134531139746)),
 (2, np.float64(1.706134531139746)),
 (3, np.float64(1.706134531139746)),
 (4, np.float64(1.706134531139746))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=REDUCE)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

16 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18), ('what', 10)]),
 (1, [('a', 2), ('it', 18)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.006121188151372969)),
   (None, np.float64(0.010492510796123566)),
   (None, np.float64(0.2061780088172206)),
   (None, np.float64(0.2134578631085825)),
   (None, np.float64(0.30080635412346945)),
   (None, np.float64(0.30239964290170573)),
   (None, np.float64(0.31007657377077824)),
   (None, np.float64(0.3246311530935415)),
   (None, np.float64(0.3280957768002688)),
   (None, np.float64(0.33669190906944635)),
   (None, np.float64(0.35803527290370585)),
   (None, np.float64(0.42670207995963105)),
   (None, np.float64(0.4549995933663712)),
   (None, np.float64(0.464328688304836))]),
 (1,
  [(None, np.float64(0.5651932904750304)),
   (None, np.float64(0.5769323147896158)),
   (None, np.float64(0.6094498684127428)),
   (None, np.float64(0.6483318359095793)),
   (None, np.float64(0.6688036074979529)),
   (None, np.float64(0.6844694372825358)),
   (None, np.float64(0.7620670618879681)),
   (None, np.float64(0.7717346547900484)),
   (None, np.float64(0.7765953306

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
input = [1, 2, 7, 4, 6, 0, -4, 1, 2]

def MAP(key, value:int):
  yield (key, value)

def REDUCE(key, values):
  yield (key, max(values))

def RECORDREADER():
  for element in input:
      yield (0, element)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 7)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
input = [1, 2, 7, 4, 6, 0, -4, 1, 2]

def MAP(key, value:int):
  yield (key, value)

def REDUCE(key, values):
  yield (key, sum(values)/len(values))

def RECORDREADER():
  for element in input:
      yield (0, element)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 2.111111111111111)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupByKey(iterable):
  t = {}
  sorted_iterable = sorted(iterable, key=lambda x:x[0])

  current_key = None # ключ текущего "отрезка" отсортированного итератора
  for (k2, v2) in sorted_iterable:
    if k2 != current_key:
      t[k2] = [v2]
      current_key = k2
    else:
      t[k2] += [v2]
  return t.items()

Проверим на примере:

In [24]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def RECORDREADER():
  return [(u.id, u) for u in data]

data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
]

list(groupByKey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female'),
   User(id=5, age=25, social_contacts=400, gender='female')]),
 (33,
  [User(id=3, age=33, social_contacts=800, gender='female'),
   User(id=4, age=33, social_contacts=900, gender='female')])]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [25]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
    User(id=4, age=33, gender='female', social_contacts=900),
]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for user in split:
        yield (user.id, user)

  split_size =  int(np.ceil(len(data)/maps))
  for i in range(0, len(data), split_size):
    yield RECORDREADER(data[i:i+split_size])

def MAP(userId, user):
  yield (userId, user)

def REDUCE(userId, users):
  unique_users = list(set(users))
  for user in unique_users:
      yield (userId, user)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=REDUCE)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

7 key-value pairs were sent over a network.


[(0,
  [(0, User(id=0, age=55, social_contacts=20, gender='male')),
   (2, User(id=2, age=25, social_contacts=500, gender='female')),
   (4, User(id=4, age=33, social_contacts=900, gender='female'))]),
 (1,
  [(1, User(id=1, age=25, social_contacts=240, gender='female')),
   (3, User(id=3, age=33, social_contacts=800, gender='female')),
   (5, User(id=5, age=25, social_contacts=400, gender='female'))])]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [26]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
]

def MAP(_, row:NamedTuple):
  if (row.social_contacts > 499):
    yield (row, row)

def REDUCE(key, value):
  yield (key, value)

def RECORDREADER():
  return [(u.id, u) for u in data]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(User(id=2, age=25, social_contacts=500, gender='female'),
  [User(id=2, age=25, social_contacts=500, gender='female')]),
 (User(id=3, age=33, social_contacts=800, gender='female'),
  [User(id=3, age=33, social_contacts=800, gender='female')]),
 (User(id=4, age=33, social_contacts=900, gender='female'),
  [User(id=4, age=33, social_contacts=900, gender='female')])]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [27]:
class Projection(NamedTuple):
  id: int
  age: int

data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
]

def MAP(_, row:NamedTuple):
  new_row = Projection(row.id, row.age)
  yield (new_row, new_row)

def REDUCE(key, values):
  yield (key, key)

def RECORDREADER():
  return [(u.id, u) for u in data]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(Projection(id=0, age=55), Projection(id=0, age=55)),
 (Projection(id=1, age=25), Projection(id=1, age=25)),
 (Projection(id=2, age=25), Projection(id=2, age=25)),
 (Projection(id=3, age=33), Projection(id=3, age=33)),
 (Projection(id=4, age=33), Projection(id=4, age=33)),
 (Projection(id=5, age=25), Projection(id=5, age=25))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [28]:
data1 = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
]

data2 = [
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
]

data = [data1, data2]

def MAP(_, row:NamedTuple):
  yield (row, row)

def REDUCE(key, value):
  yield (key, key)

def RECORDREADER():
    return [(u.id, u) for part in data for u in part]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(User(id=0, age=55, social_contacts=20, gender='male'),
  User(id=0, age=55, social_contacts=20, gender='male')),
 (User(id=1, age=25, social_contacts=240, gender='female'),
  User(id=1, age=25, social_contacts=240, gender='female')),
 (User(id=2, age=25, social_contacts=500, gender='female'),
  User(id=2, age=25, social_contacts=500, gender='female')),
 (User(id=3, age=33, social_contacts=800, gender='female'),
  User(id=3, age=33, social_contacts=800, gender='female')),
 (User(id=4, age=33, social_contacts=900, gender='female'),
  User(id=4, age=33, social_contacts=900, gender='female')),
 (User(id=5, age=25, social_contacts=400, gender='female'),
  User(id=5, age=25, social_contacts=400, gender='female'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [29]:
data1 = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
]

data2 = [
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
]

data = [data1, data2]

def MAP(_, row:NamedTuple):
  yield (row, row)

def REDUCE(key, value):
  if len(value)>1:
    yield (key, key)

def RECORDREADER():
  return [(u.id, u) for part in data for u in part]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(User(id=2, age=25, social_contacts=500, gender='female'),
  User(id=2, age=25, social_contacts=500, gender='female')),
 (User(id=3, age=33, social_contacts=800, gender='female'),
  User(id=3, age=33, social_contacts=800, gender='female'))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [30]:
data1 = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
]

data2 = [
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=33, gender='female', social_contacts=900),
    User(id=5, age=25, gender='female', social_contacts=400),
]

data = [data1, data2]

def MAP(key, row:NamedTuple):
  yield (row, key)

def REDUCE(key, value):
  if value == [0]:
    yield (key, key)

def RECORDREADER():
  for index, part in enumerate(data):
    for user in part:
      yield (index, user) # индекс списка + юзер

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(User(id=0, age=55, social_contacts=20, gender='male'),
  User(id=0, age=55, social_contacts=20, gender='male')),
 (User(id=1, age=25, social_contacts=240, gender='female'),
  User(id=1, age=25, social_contacts=240, gender='female'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [31]:
class UserAge(NamedTuple):
  id: int
  age: int

class UserGender(NamedTuple):
  id: int
  gender: int

data1 = [
    UserAge(id=0, age=55),
    UserAge(id=1, age=25),
    UserAge(id=2, age=25),
    UserAge(id=3, age=37),
]

data2 = [
    UserGender(id=0, gender='male'),
    UserGender(id=1, gender='female'),
    UserGender(id=2, gender='female'),
    UserGender(id=4, gender='female'),
]

data = [data1, data2]

def MAP(key, row:NamedTuple):
  if key == 0: # отношение R
    yield (row.id, (0, row.age))
  else:
    yield (row.id, (1, row.gender))


def REDUCE(b, values):
  r_values = [a for (index, a) in values if index == 0]
  s_values = [c for (index, c) in values if index == 1]
  for a in r_values:
    for c in s_values:
      yield(a,b,c)

def RECORDREADER():
  for index, part in enumerate(data):
    for user in part:
      yield (index, user) # индекс списка + юзер

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(55, 0, 'male'), (25, 1, 'female'), (25, 2, 'female')]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [32]:
class MaxAgeByGender(NamedTuple):
  groupId: int
  gender: str
  max_age: int

class AgeGender(NamedTuple):
  id : int
  gender: str
  age: int

data = [
    AgeGender(id=0, gender='male', age=25),
    AgeGender(id=1, gender='male', age=67),
    AgeGender(id=2, gender='female', age=18),
    AgeGender(id=3, gender='male', age=52),
    AgeGender(id=4, gender='female', age=38),
    AgeGender(id=5, gender='female', age=65),
]

def MAP(key, row:NamedTuple):
  yield(row.gender, row.age)


def REDUCE(key, values):
  yield (key, max(values))


def RECORDREADER():
  return [(u.id, u) for u in data]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output


[('male', 67), ('female', 65)]

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [45]:
mat = np.ones((5,4))
#vec = np.random.rand(4)
vec = np.array([1,2,3,4])

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(start_row, end_row):
    for i in range(start_row, end_row):
      for j in range(mat.shape[1]): # количество колонок
        yield ((i, j), (mat[i, j], vec[j]))

  split_size = int(np.ceil(len(mat) / maps))


  for i in range(0, maps):
    start_row = i * split_size
    end_row = min(start_row + split_size, mat.shape[0])
    yield RECORDREADER(start_row, end_row)


def MAP(coordinates: (int, int), values):
  i, j = coordinates
  matrix_value, vector_value = values
  yield (i, matrix_value * vector_value)


def REDUCE(i, products: Iterator[NamedTuple]):
  yield (i, sum(products))


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

20 key-value pairs were sent over a network.


[(0, [(0, np.float64(10.0)), (2, np.float64(10.0)), (4, np.float64(10.0))]),
 (1, [(1, np.float64(10.0)), (3, np.float64(10.0))])]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [46]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [50]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1 # координаты значения большой матрицы
  w = v1 # значение большой матрицы
  for i in range(I):
    yield ((i, k), w * small_mat[i,j])

def REDUCE(key, values):
  (i, k) = key
  yield ((i,k), sum(values))

Проверьте своё решение

In [51]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [52]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [54]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      for i in range(small_mat.shape[0]):
        yield (((i,j), small_mat[i,j]), ((j,k), big_mat[j,k]))

def MAP(mat1_tuple, mat2_tuple):
  k1, w1 = mat1_tuple
  k2, w2 = mat2_tuple
  i = k1[0]
  k = k2[1]

  yield ((i, k), w1 * w2)

def REDUCE(key, values):
  (i, k) = key
  yield ((i,k), sum(values))

In [55]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [76]:
import numpy as np
I = 2
J = 3
K = 4*10

maps = 2
reducers = 3

small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def INPUTFORMAT():
  def RECORDREADER(key, split):
    for i in range(split.shape[0]):
      for j in range(split.shape[1]):
        yield ((i,j), (key, split[i,j]))

  yield RECORDREADER("S", small_mat)
  yield RECORDREADER("B", big_mat)


def MAP(key, value):
  (i, j), (mat_type, w) = key, value
  if mat_type == 'S':
    yield (j, (mat_type, i, w))
  else:
    yield (i, (mat_type, j, w))

def REDUCE(key, values):
  S_rows = [v for v in values if v[0] == 'S']
  B_cols = [v for v in values if v[0] == 'B']

  for s_row in S_rows:
    for b_col in B_cols:
      i,k = s_row[1], b_col[1]
      yield ((i,k), s_row[2] * b_col[2])

def INPUT_MUL():
  for j in joined:
      yield j[1]

def MAP_MUL(k1, v1):
  yield (k1, v1)

def REDUCE_MUL(key, values):  # финальная сумма
    yield (key, sum(values))


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(INPUT_MUL, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
    for v in p[1]:
        solution.append(v)


126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.


In [77]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
#solution = MapReduceDistributed(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [78]:
import numpy as np
I = 2
J = 3
K = 4*10
N_S, N_B = 2, 3  # количество RECORDREADER-ов для S и B

maps = 2
reducers = 3

small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def INPUTFORMAT():
  def RECORDREADER(key, split, start_row, end_row):
    for i in range(start_row, end_row):
      for j in range(split.shape[1]):
        yield ((i,j), (key, split[i,j]))

  split_size_S = int(np.ceil(I / N_S))
  for reader in range(N_S):
    start_row = reader * split_size_S
    end_row = min(start_row + split_size_S, I)
    yield RECORDREADER("S", small_mat, start_row, end_row)

  split_size_B = int(np.ceil(J / N_B))
  for reader in range(N_B):
    start_row = reader * split_size_B
    end_row = min(start_row + split_size_B, J)
    yield RECORDREADER("B", big_mat, start_row, end_row)



def MAP(key, value):
  (i, j), (mat_type, w) = key, value
  if mat_type == 'S':
    yield (j, (mat_type, i, w))
  else:
    yield (i, (mat_type, j, w))

def REDUCE(key, values):
  S_rows = [v for v in values if v[0] == 'S']
  B_cols = [v for v in values if v[0] == 'B']

  for s_row in S_rows:
    for b_col in B_cols:
      i,k = s_row[1], b_col[1]
      yield ((i,k), s_row[2] * b_col[2])

def INPUT_MUL():
  for j in joined:
      yield j[1]

def MAP_MUL(k1, v1):
  yield (k1, v1)

def REDUCE_MUL(key, values):  # финальная сумма
    yield (key, sum(values))


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(INPUT_MUL, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
    for v in p[1]:
        solution.append(v)

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.


In [79]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
#solution = MapReduceDistributed(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Решение не будет работать, если каждый RECORDREADER будет генерировать случайное подмножество элементов матрицы, потому что в таком случае возможны повторы или пропуски элементов матрицы, а для корректного подсчета произведения нужны все элементы с правильными индексами.